# Bundle Loading Guide Notebook

Mirrors the [Bundle Loading Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_loading_guide/). Pair with the [authoring notebook](bundle_authoring.ipynb): the bundle this opens (`experiment-1.sdsge`) is produced there.

In [1]:
import numpy as np

from SymbolicDSGE import load_bundle
from SymbolicDSGE.core.solved_model import SolvedModel
from SymbolicDSGE.estimation.results import MCMCResult, OptimizationResult
from SymbolicDSGE.monte_carlo import validate_pipeline_spec

from typing import cast

## Open the bundle

In [2]:
loaded = load_bundle("experiment-1.sdsge")

print("Created by:", loaded.manifest.created_by)
print("Created at:", loaded.manifest.created_at)
print("Format version:", loaded.manifest.sdsge_version)

Created by: experiment-1
Created at: 2026-07-05T22:10:25.100954+00:00
Format version: 1


## Reach the solved models

In [3]:
# Cast so type checker doesn't think it can be `None`; completely optional.
reference = cast(SolvedModel, loaded.reference)
dgp = cast(SolvedModel, loaded.dgp)

# The `if` check narrows the type if casting is not preferred.
if reference is not None:
    print("Stable:", reference.policy.stab == 0)
    print("Eigenvalues:", reference.policy.eig.round(2), "\n")

if dgp is not None:
    print("Stable:", dgp.policy.stab == 0)
    print("Eigenvalues:", dgp.policy.eig.round(2))

Stable: True
Eigenvalues: [0.28+0.j 0.83+0.j 0.85+0.j 2.6 +0.j 1.19+0.j] 

Stable: True
Eigenvalues: [0.28+0.j 0.83+0.j 0.85+0.j 2.6 +0.j 1.19+0.j]


In [4]:
T = 20
rng = np.random.default_rng(42)
shocks = {
    "g,z": rng.standard_normal((T, 2)),
}
sim = reference.sim(
    T=T,
    shocks=shocks,
    observables=True,
)
print(sim["Infl"][:5])

[  3.43         8.20387037  10.83531974 -12.17755328   0.91492503]


## Reach the estimation tab

In [5]:
estimation = loaded.estimation

if estimation is not None:
    print("Method:", estimation.spec.method)
    print("Observables:", estimation.spec.observables)
    print("Parameters:", [p.name for p in estimation.spec.parameters])

Method: mcmc
Observables: ['OutGap', 'Infl', 'Rate']
Parameters: ['psi_pi', 'psi_x']


In [6]:
result = estimation.result
if isinstance(result, OptimizationResult):
    print("Point estimate:", result.theta)
    print("Log-posterior:", result.logpost)
elif isinstance(result, MCMCResult):
    print("Acceptance:", round(result.accept_rate, 2))
    print("Draws:", result.n_draws, "burn-in:", result.burn_in)

Acceptance: 0.41
Draws: 1000 burn-in: 200


In [7]:
if estimation.observed is not None:
    print("Observed shape:", estimation.observed.shape)
if estimation.posterior is not None:
    samples = estimation.posterior["samples"]
    print("Posterior mean:", samples.mean(axis=0))

Observed shape: (40, 3)
Posterior mean: [1.91493782 0.72163645]


Re-run the estimation from the loaded spec to produce a *fresh* result object for reproduction, or when a bundle stored the spec without a result. When results **were** bundled, `estimation.result` already holds the live object, so this is not needed for inspection:

```python
inputs = estimation.spec.to_estimator_inputs()

solver = DSGESolver(loaded.reference.config, loaded.reference.kalman_config)
extras: dict = {}
if inputs.bounds is not None and estimation.spec.method != "mcmc":
    extras["bounds"] = inputs.bounds  # MLE/MAP only

fresh_result = solver.estimate(
    compiled=loaded.reference.compiled,
    y=estimation.observed,
    method=estimation.spec.method,
    estimated_params=inputs.estimated_params,
    theta0=inputs.theta0,
    priors=inputs.priors,
    observables=estimation.spec.observables,
    **extras,
)
print("Re-run result:", type(fresh_result).__name__)
```

### Run an estimation from a loaded bundle

`spec.to_estimator_inputs()` lowers the loaded spec into the concrete arguments `DSGESolver.estimate` expects: `Prior` objects built from each `PriorSpec`, parameter initials, and (for MLE/MAP) bounds.

In [8]:
inputs = estimation.spec.to_estimator_inputs()
inputs

EstimatorInputs(estimated_params=['psi_pi', 'psi_x'], theta0={'psi_pi': 2.19, 'psi_x': 0.3}, priors={'psi_pi': Prior(dist=Normal, transform=Identity), 'psi_x': Prior(dist=Normal, transform=Identity)}, bounds=None)

In [9]:
# The loaded MCMC result is already a live MCMCResult. Call diagnostics on it
# directly. No rebuild from metadata and posterior traces is needed.
if isinstance(result, MCMCResult):
    print(result.hpd_intervals(alpha=0.05))

{'psi_pi': (np.float64(1.5226551191377002), np.float64(2.350789392221446)), 'psi_x': (np.float64(0.37114074135853414), np.float64(1.0535896356712162))}


## Reach the Monte Carlo tab

In [10]:
mc = loaded.mc

if mc is not None:
    print("Runtime Steps:", [step.name for step in mc.pipeline.per_rep_steps])
    print("Post-Processing:", [step.name for step in mc.pipeline.postproc_steps])

Runtime Steps: ['datagen', 'jb_test']
Post-Processing: []


### Run a Monte Carlo pipeline from a loaded bundle

`LoadedMC.pipeline` is a live `MCPipeline` rebuilt during `load_bundle(...)`. Run it directly against the loaded models. No `[ui]` extra is required.

In [11]:
# The pipeline's simulation datagen needs a DGP. The authoring notebook
# bundles a model under role "dgp", so `loaded.dgp` resolves. If a bundle
# omits `reference` or `dgp`, provide the missing model when running again.
n_rep = mc.document["n_rep"]
mc_result = mc.pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=n_rep,
    retain_payloads=False,
    retain_test_results=False,
    retain_contexts=False,
    fail_fast=True,
    verbosity=1,
)
print("Successful reps:", mc_result.n_successful, "/", mc_result.n_rep)

jb = mc_result.test_summaries["jb_test"]
stat_ci = jb.statistic_confidence_interval(0.95)
pval_ci = jb.pval_confidence_interval(0.95)

list(zip(stat_ci, pval_ci))

MC run concluded successfully in 1.74s with 576.18 it/s.
Successful reps: 1000 / 1000


[(np.float64(883.1437534372493), np.float64(0.8170364534597756)),
 (np.float64(1943.6351442634336), np.float64(0.8623536972530759))]

## Reach the simulation prefill

In [16]:
prefills = loaded.simulation  # dict[str, SimSpec] | None

if prefills is not None:
    for role, spec in prefills.items():
        print(role, "\n", spec)

reference.sim(**prefills["reference"])["r"][:5]

reference 
 SimSpec(T=200, x0=None, observables=True, shock_scale=1.0, shocks={'r': {'dist': 'norm', 'multivar': False, 'seed': 42, 'dist_args': [], 'dist_kwargs': {'loc': 0.0}}})


array([ 0.        ,  0.05484907, -0.17182928,  0.08693731,  0.19366014])